In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd

from config import RAW_DATA_DIR, PROCESSED_DATA_DIR

In [2]:
icustays_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "icustays_clean.parquet",
    columns=["subject_id", "hadm_id", "stay_id", "intime", "outtime"]
)

icustays_df["intime"] = pd.to_datetime(icustays_df["intime"])
icustays_df["outtime"] = pd.to_datetime(icustays_df["outtime"])

print(icustays_df.shape)
print(icustays_df["stay_id"].duplicated().sum())

(94444, 5)
0


In [3]:
d_items_df = pd.read_csv(RAW_DATA_DIR / "d_items.csv")

input_items_df = d_items_df[d_items_df["linksto"].eq("inputevents")][
    ["itemid", "label", "category", "unitname", "param_type"]
].copy()

print(input_items_df.shape)
input_items_df.head()

(476, 5)


,itemid,label,category,unitname,param_type
162,220861,Albumin (Human) 20%,Fluids - Other (Not In Use),mL,Solution
163,220862,Albumin 25%,Blood Products/Colloids,mL,Solution
164,220863,Albumin (Human) 4%,Fluids - Other (Not In Use),mL,Solution
165,220864,Albumin 5%,Blood Products/Colloids,mL,Solution
166,220865,Aquadestila,Fluids - Other (Not In Use),mL,Solution


In [4]:
def classify_input_group(label, category, ordercategoryname):
    text = " ".join(
        str(value).upper()
        for value in [label, category, ordercategoryname]
        if pd.notna(value)
    )

    if any(term in text for term in ["NOREPINEPHRINE", "EPINEPHRINE", "PHENYLEPHRINE", "VASOPRESSIN", "DOPAMINE", "DOBUTAMINE", "MILRINONE"]):
        return "vasopressor_inotrope"

    elif "INSULIN" in text:
        return "insulin"

    elif any(term in text for term in ["PROPOFOL", "MIDAZOLAM", "VERSED", "FENTANYL", "DEXMEDETOMIDINE", "PRECEDEX", "MORPHINE"]):
        return "sedative_analgesic"

    elif "HEPARIN" in text:
        return "anticoagulant"

    elif any(term in text for term in ["BLOOD PRODUCTS", "PACKED RED", "RED BLOOD", "PLATELET", "FRESH FROZEN", "WHOLE BLOOD", "CRYO"]):
        return "blood_product"

    elif "ALBUMIN" in text or "COLLOID" in text:
        return "colloid"

    elif any(term in text for term in ["CRYSTALLOID", "SALINE", "RINGER", "LACTATED", "DEXTROSE"]):
        return "crystalloid"

    elif "ENTERAL NUTRITION" in text:
        return "enteral_nutrition"

    elif "PARENTERAL NUTRITION" in text or "PN" in text:
        return "parenteral_nutrition"

    elif "ANTIBIOTIC" in text:
        return "antibiotic"

    elif "ORAL/GASTRIC" in text:
        return "oral_gastric"

    elif "FLUID" in text or "INTAKE" in text:
        return "other_fluid"

    elif "MEDICATION" in text or "MED" in text:
        return "other_medication"

    else:
        return "other"


def normalize_unit(unit):
    if pd.isna(unit):
        return "unknown"

    unit = str(unit).strip().lower()
    unit = unit.replace("/", "_per_").replace(" ", "_").replace(".", "")
    unit = unit.replace("international_units", "iu")

    unit_map = {
        "ml": "ml",
        "l": "l",
        "mg": "mg",
        "mcg": "mcg",
        "units": "units",
        "dose": "dose",
        "meq": "meq",
        "meq": "meq",
        "grams": "grams",
        "mmol": "mmol",
        "ounces": "ounces",
    }

    return unit_map.get(unit, unit)

In [5]:
usecols = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "starttime",
    "endtime",
    "storetime",
    "itemid",
    "amount",
    "amountuom",
    "rate",
    "rateuom",
    "orderid",
    "linkorderid",
    "ordercategoryname",
    "ordercategorydescription",
    "patientweight",
    "totalamount",
    "totalamountuom",
    "statusdescription",
]

chunksize = 1_000_000
stay_ids = set(icustays_df["stay_id"])
clean_chunks = []

for chunk in pd.read_csv(
    RAW_DATA_DIR / "inputevents.csv",
    usecols=usecols,
    chunksize=chunksize,
):
    chunk = chunk[
        chunk["stay_id"].isin(stay_ids)
        & chunk["amount"].notna()
        & chunk["amount"].gt(0)
    ].copy()

    if chunk.empty:
        continue

    chunk["starttime"] = pd.to_datetime(chunk["starttime"])
    chunk["endtime"] = pd.to_datetime(chunk["endtime"])
    chunk["storetime"] = pd.to_datetime(chunk["storetime"])

    chunk = chunk.merge(
        icustays_df[["stay_id", "intime", "outtime"]],
        on="stay_id",
        how="inner"
    )

    chunk = chunk[
        chunk["starttime"].le(chunk["outtime"])
        & chunk["endtime"].ge(chunk["intime"])
    ].copy()

    chunk["icu_overlap_starttime"] = chunk[["starttime", "intime"]].max(axis=1)
    chunk["icu_overlap_endtime"] = chunk[["endtime", "outtime"]].min(axis=1)
    chunk["duration_hours"] = (
        (chunk["endtime"] - chunk["starttime"])
        .dt.total_seconds()
        .div(3600)
        .clip(lower=0)
    )
    chunk["icu_overlap_duration_hours"] = (
        (chunk["icu_overlap_endtime"] - chunk["icu_overlap_starttime"])
        .dt.total_seconds()
        .div(3600)
        .clip(lower=0)
    )
    chunk["hours_from_icu_admit_start"] = (
        (chunk["icu_overlap_starttime"] - chunk["intime"])
        .dt.total_seconds()
        .div(3600)
    )
    chunk["hours_from_icu_admit_end"] = (
        (chunk["icu_overlap_endtime"] - chunk["intime"])
        .dt.total_seconds()
        .div(3600)
    )

    chunk = chunk.merge(
        input_items_df,
        on="itemid",
        how="left"
    )

    chunk["input_label"] = chunk["label"].fillna("Unknown")
    chunk["input_category"] = chunk["category"].fillna("Unknown")
    chunk["input_group"] = chunk.apply(
        lambda row: classify_input_group(row["input_label"], row["input_category"], row["ordercategoryname"]),
        axis=1
    )
    chunk["amountuom_clean"] = chunk["amountuom"].apply(normalize_unit)
    chunk["rateuom_clean"] = chunk["rateuom"].apply(normalize_unit)

    clean_chunks.append(
        chunk[
            [
                "subject_id",
                "hadm_id",
                "stay_id",
                "starttime",
                "endtime",
                "storetime",
                "icu_overlap_starttime",
                "icu_overlap_endtime",
                "hours_from_icu_admit_start",
                "hours_from_icu_admit_end",
                "duration_hours",
                "icu_overlap_duration_hours",
                "itemid",
                "input_label",
                "input_category",
                "input_group",
                "amount",
                "amountuom",
                "amountuom_clean",
                "rate",
                "rateuom",
                "rateuom_clean",
                "orderid",
                "linkorderid",
                "ordercategoryname",
                "ordercategorydescription",
                "patientweight",
                "totalamount",
                "totalamountuom",
                "statusdescription",
            ]
        ]
    )

if clean_chunks:
    inputevents_clean_df = pd.concat(clean_chunks, ignore_index=True)
else:
    inputevents_clean_df = pd.DataFrame(columns=[
        "subject_id",
        "hadm_id",
        "stay_id",
        "starttime",
        "endtime",
        "storetime",
        "icu_overlap_starttime",
        "icu_overlap_endtime",
        "hours_from_icu_admit_start",
        "hours_from_icu_admit_end",
        "duration_hours",
        "icu_overlap_duration_hours",
        "itemid",
        "input_label",
        "input_category",
        "input_group",
        "amount",
        "amountuom",
        "amountuom_clean",
        "rate",
        "rateuom",
        "rateuom_clean",
        "orderid",
        "linkorderid",
        "ordercategoryname",
        "ordercategorydescription",
        "patientweight",
        "totalamount",
        "totalamountuom",
        "statusdescription",
    ])

print(inputevents_clean_df.shape)
print(inputevents_clean_df.isna().sum())
print(inputevents_clean_df["input_group"].value_counts())

(10918301, 30)
subject_id                          0
hadm_id                             0
stay_id                             0
starttime                           0
endtime                             0
storetime                           0
icu_overlap_starttime               0
icu_overlap_endtime                 0
hours_from_icu_admit_start          0
hours_from_icu_admit_end            0
duration_hours                      0
icu_overlap_duration_hours          0
itemid                              0
input_label                         0
input_category                      0
input_group                         0
amount                              0
amountuom                           0
amountuom_clean                     0
rate                          4875171
rateuom                       4875171
rateuom_clean                       0
orderid                             0
linkorderid                         0
ordercategoryname                   0
ordercategorydescription           

In [6]:
inputevents_clean_df = inputevents_clean_df.sort_values(
    ["stay_id", "input_group", "icu_overlap_starttime"]
).reset_index(drop=True)

print(inputevents_clean_df.duplicated().sum())
print(inputevents_clean_df[["hours_from_icu_admit_start", "hours_from_icu_admit_end", "amount", "duration_hours"]].describe())

0
       hours_from_icu_admit_start  hours_from_icu_admit_end        amount  \
count                1.091830e+07              1.091830e+07  1.091830e+07   
mean                 1.325075e+02              1.347394e+02  2.128740e+02   
std                  1.942119e+02              1.945049e+02  1.296279e+03   
min                  0.000000e+00              0.000000e+00  2.136027e-05   
25%                  1.910222e+01              2.098333e+01  2.847235e+00   
50%                  6.190306e+01              6.426667e+01  2.992600e+01   
75%                  1.687094e+02              1.713228e+02  1.000000e+02   
max                  3.826526e+03              3.831976e+03  1.000400e+06   

       duration_hours  
count    1.091830e+07  
mean     2.233977e+00  
std      5.503021e+00  
min      1.666667e-02  
25%      1.666667e-02  
50%      4.166667e-01  
75%      2.016667e+00  
max      2.787833e+02  


In [7]:
inputevents_clean_df.to_parquet(
    PROCESSED_DATA_DIR / "inputevents_clean.parquet",
    index=False
)